In [1]:
import pandas as pd


In [2]:
df = pd.read_csv(r"C:\Users\PRITESH\Downloads\archive (48)\ab_data.csv")

In [3]:
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [4]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       294478 non-null  int64 
 1   timestamp     294478 non-null  object
 2   group         294478 non-null  object
 3   landing_page  294478 non-null  object
 4   converted     294478 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.2+ MB


user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64

In [5]:
df = df.drop_duplicates(subset='user_id')

In [6]:
df_mismatch = df[
    ((df['group'] == 'control') & (df['landing_page'] == 'new_page')) |
    ((df['group'] == 'treatment') & (df['landing_page'] == 'old_page'))
]

In [7]:
df_mismatch


,user_id,timestamp,group,landing_page,converted
22,767017,2017-01-12 22:58:14.991443,control,new_page,0
240,733976,2017-01-11 15:11:16.407599,control,new_page,0
308,857184,2017-01-20 07:34:59.832626,treatment,old_page,0
327,686623,2017-01-09 14:26:40.734775,treatment,old_page,0
357,856078,2017-01-12 12:29:30.354835,treatment,old_page,0
...,...,...,...,...,...
282645,717723,2017-01-10 18:43:58.148113,treatment,old_page,0
282950,778907,2017-01-04 02:54:16.589129,control,new_page,0
285987,914482,2017-01-05 16:55:40.060421,treatment,old_page,0
286353,767924,2017-01-08 21:48:53.436050,treatment,old_page,0


In [8]:
df = df[
    ((df['group'] == 'control') & (df['landing_page'] == 'old_page')) |
    ((df['group'] == 'treatment') & (df['landing_page'] == 'new_page'))
]

In [9]:
df

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1
...,...,...,...,...,...
294473,751197,2017-01-03 22:28:38.630509,control,old_page,0
294474,945152,2017-01-12 00:51:57.078372,control,old_page,0
294475,734608,2017-01-22 11:45:03.439544,control,old_page,0
294476,697314,2017-01-15 01:20:28.957438,control,old_page,0


In [10]:
df['group'].value_counts(normalize=True)

group
treatment    0.500152
control      0.499848
Name: proportion, dtype: float64

In [11]:
df['converted'].value_counts()

converted
0    254057
1     34483
Name: count, dtype: int64

In [12]:
df.groupby(['group', 'landing_page']).size()

group      landing_page
control    old_page        144226
treatment  new_page        144314
dtype: int64

In [13]:
df.to_csv("clean_ab_data.csv", index=False)

# EDA

In [14]:
df.groupby('group')['converted'].mean()

group
control      0.120290
treatment    0.118727
Name: converted, dtype: float64

In [15]:
df['group'].value_counts()

group
treatment    144314
control      144226
Name: count, dtype: int64

In [16]:
df.groupby('group')['converted'].sum()

group
control      17349
treatment    17134
Name: converted, dtype: int64

In [17]:
#Checking which is best if lift is postive new_page is best or if lift is negative so old_page is best

cr=df.groupby('landing_page')['converted'].mean()

lift = (cr['new_page']-cr['old_page'])/cr['old_page']*100


lift 

np.float64(-1.299486975292441)

### Insight - 
As we see a in our case lift is negative so old_page is good but is not statistical proof so now we perfrom hypothesis testing

In [34]:
# Here we used Two-proportion Z-test is Compares the success rates (proportions) of two independent groups.


from statsmodels.stats.proportion import proportions_ztest


conversions = df.groupby('landing_page')['converted'].sum()
users = df.groupby('landing_page')['converted'].count()
z_stat, p_value = proportions_ztest(conversions, users)

# significance level
alpha = 0.05

# hypothesis decision
if p_value < alpha:
    print("Reject Null Hypothesis (H0)")
    print("Conclusion: There is a statistically significant difference between landing_page.")
else:
    print("Fail to Reject Null Hypothesis (H0)")
    print("Conclusion: No statistically significant difference between landing_page.")

Fail to Reject Null Hypothesis (H0)
Conclusion: No statistically significant difference between landing_page.


landing_page
new_page    17134
old_page    17349
Name: converted, dtype: int64

### “The analysis shows that the new landing page does not significantly improve conversion rate compared to the old page. Therefore, it is not recommended to implement the new design.”